# Disagreement-Aware Probabilistic U-Net on LIDC

Selected paper: Kohl et al., *A Probabilistic U-Net for Segmentation of Ambiguous Images*, NeurIPS 2018.

Extension: predict the human inter-rater disagreement map from the image and align stochastic sample diversity with the pixels where annotators disagree.

## Dependencies

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = ""  # Optional: set to your GitHub repo URL when opening the notebook directly in Colab.
REPO_DIR = Path("/content/probabilistic-unet") if "google.colab" in sys.modules else Path.cwd()

if not (Path.cwd() / "setup.py").exists():
    if not REPO_URL:
        raise RuntimeError(
            "Repository files are not present. In Colab, set REPO_URL to the project repo URL "
            "or upload/clone the repository before running this cell."
        )
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
elif "google.colab" in sys.modules and Path.cwd() != REPO_DIR:
    REPO_DIR = Path.cwd()

REPO_ROOT = Path.cwd()

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

## Configuration

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

import torch
from IPython.display import Image as DisplayImage, Markdown, display

RUN_MODE = "smoke"  # "smoke", "pilot", or "final"
FORCE_RETRAIN = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("data/lidc")
OUTPUT_ROOT = Path("outputs_notebook") / RUN_MODE
SEED = 1

MODE_CONFIGS = {
    "smoke": {
        "steps": 2,
        "max_train": 8,
        "max_val": 4,
        "max_test": 4,
        "batch_size": 1,
        "eval_batch_size": 1,
        "feature_maps": 2,
        "latent_size": 2,
        "depth": 2,
        "train_samples": 2,
        "eval_samples": 2,
        "eval_every": 1,
        "save_every": 1,
        "figure_cases": 2,
        "figure_samples": 2,
    },
    "pilot": {
        "steps": 10000,
        "max_train": None,
        "max_val": 256,
        "max_test": 256,
        "batch_size": 16,
        "eval_batch_size": 8,
        "feature_maps": 16,
        "latent_size": 6,
        "depth": 4,
        "train_samples": 4,
        "eval_samples": 16,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
    "final": {
        "steps": 100000,
        "max_train": None,
        "max_val": None,
        "max_test": None,
        "batch_size": 32,
        "eval_batch_size": 8,
        "feature_maps": 32,
        "latent_size": 6,
        "depth": 5,
        "train_samples": 4,
        "eval_samples": 16,
        "eval_every": 1000,
        "save_every": 1000,
        "figure_cases": 8,
        "figure_samples": 4,
    },
}
CFG = MODE_CONFIGS[RUN_MODE]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"mode={RUN_MODE} device={DEVICE} output={OUTPUT_ROOT}")

MAIN_VARIANTS = ("baseline", "head", "full")

def run(cmd, *, check=True):
    print("\n$ " + " ".join(map(str, cmd)))
    return subprocess.run([str(x) for x in cmd], check=check)

def maybe_arg(name, value):
    return [] if value is None else [name, str(value)]

## Data

In [ ]:
run([sys.executable, "scripts/download_lidc.py", "--dest", DATA_ROOT])

## Human Disagreement Preview

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from probunet.disagreement import compute_disagreement
from probunet.lidc import LIDCCrops

preview_dir = OUTPUT_ROOT / "preview"
preview_dir.mkdir(parents=True, exist_ok=True)
ds_preview = LIDCCrops(root=DATA_ROOT, split="val", train=False, single_random_grader=False)
sample = ds_preview[0]
d_map = compute_disagreement(sample["masks"][None]).numpy()[0, 0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.4))
panels = [sample["image"][0].numpy()] + [sample["masks"][i].numpy() for i in range(4)] + [d_map]
titles = ["image", "mask 1", "mask 2", "mask 3", "mask 4", "human D"]
for ax, panel, title in zip(axes, panels, titles):
    cmap = "magma" if title == "human D" else "gray"
    ax.imshow(panel, cmap=cmap, vmin=0 if cmap == "magma" else None, vmax=1 if cmap == "magma" else None)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
preview_path = preview_dir / "disagreement_preview.png"
fig.savefig(preview_path, dpi=160)
plt.close(fig)
display(DisplayImage(filename=str(preview_path)))

## Arm A-C: ELBO Ablation

In [ ]:
def train_main_variant(variant):
    out_dir = OUTPUT_ROOT / "fcombfix" / "lidc_ablation" / variant
    latest = out_dir / "latest_checkpoint.pt"
    if latest.exists() and not FORCE_RETRAIN:
        print(f"Skipping {variant}; found {latest}")
        return
    cmd = [
        sys.executable, "scripts/train_lidc_ablation.py",
        "--variant", variant,
        "--data-root", DATA_ROOT,
        "--out-dir", out_dir,
        "--device", DEVICE,
        "--steps", CFG["steps"],
        "--batch-size", CFG["batch_size"],
        "--eval-batch-size", CFG["eval_batch_size"],
        "--feature-maps", CFG["feature_maps"],
        "--latent-size", CFG["latent_size"],
        "--depth", CFG["depth"],
        "--train-samples", CFG["train_samples"],
        "--eval-samples", CFG["eval_samples"],
        "--eval-every", CFG["eval_every"],
        "--save-every", CFG["save_every"],
        "--lambda-disagreement", 0.01,
        "--lambda-alignment", 1e-5,
        "--grad-clip", 100,
        "--seed", SEED,
    ]
    cmd += maybe_arg("--max-train", CFG["max_train"])
    cmd += maybe_arg("--max-val", CFG["max_val"])
    run(cmd)

for variant in MAIN_VARIANTS:
    train_main_variant(variant)

## Test Results

In [ ]:
fcomb_runs = OUTPUT_ROOT / "fcombfix" / "lidc_ablation"
fcomb_eval = OUTPUT_ROOT / "fcombfix" / "final_eval_100k"
cmd = [
    sys.executable, "scripts/eval_lidc_final.py",
    "--runs-dir", fcomb_runs,
    "--variants", *MAIN_VARIANTS,
    "--checkpoint-name", "latest_checkpoint.pt",
    "--data-root", DATA_ROOT,
    "--batch-size", CFG["eval_batch_size"],
    "--eval-samples", CFG["eval_samples"],
    "--device", DEVICE,
    "--out-dir", fcomb_eval,
]
cmd += maybe_arg("--max-test", CFG["max_test"])
run(cmd)
display(Markdown((fcomb_eval / "summary.md").read_text()))


## Qualitative Results

In [ ]:
fig_dir = OUTPUT_ROOT / "fcombfix" / "figures_100k"
cmd = [
    sys.executable, "scripts/make_lidc_figures.py",
    "--baseline-checkpoint", fcomb_runs / "baseline" / "latest_checkpoint.pt",
    "--full-checkpoint", fcomb_runs / "full" / "latest_checkpoint.pt",
    "--data-root", DATA_ROOT,
    "--split", "test",
    "--out-dir", fig_dir,
    "--num-cases", CFG["figure_cases"],
    "--samples", CFG["figure_samples"],
    "--device", DEVICE,
]
if RUN_MODE == "smoke":
    cmd += ["--selection", "first"]
run(cmd)

figure_paths = sorted(fig_dir.glob("case_*.png"))
print(f"Generated {len(figure_paths)} qualitative figures in {fig_dir}")
for path in figure_paths[: min(4, len(figure_paths))]:
    print(path)
    display(DisplayImage(filename=str(path)))

## Architecture Panels

In [ ]:
asset_dir = OUTPUT_ROOT / "overleaf_figs"
run([
    sys.executable, "scripts/make_figure_assets.py",
    "--checkpoint", fcomb_runs / "full" / "latest_checkpoint.pt",
    "--data-root", DATA_ROOT,
    "--out-dir", asset_dir,
    "--samples", max(16, CFG["eval_samples"]),
    "--device", DEVICE,
])

for name in ["ct.png", "mask0.png", "mask1.png", "mask2.png", "mask3.png", "dgt.png", "sample0.png", "sample1.png", "sample2.png", "u.png", "dhat.png", "pred.png"]:
    path = asset_dir / name
    if path.exists():
        print(path)
        display(DisplayImage(filename=str(path), width=120))

## Generated Artifacts

- `fcombfix/lidc_ablation/<baseline|head|full>/history.csv`
- `fcombfix/lidc_ablation/<baseline|head|full>/{best,latest}_checkpoint.pt`
- `fcombfix/final_eval_100k/{test_metrics.csv,summary.md}`
- `fcombfix/figures_100k/case_*.png`
- `overleaf_figs/*.png`